# Fairness-Aware GPT-2 — full QQP run (all three models)

**Built for `Save Version → Save & Run All (Commit)`.** Runs unattended; close
your laptop.

Trains **baseline → cda → cda_reg**, evaluates each, uploads `cda_reg` to Hugging
Face, and prints the baseline-vs-fairness comparison.

Order matters: baseline first, `cda_reg` last, so if the session dies you keep
the models you care about most.

## Before you commit

Right panel → **Session options**:

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** — NOT P100 (compute 6.0, unsupported) |
| **Internet** | **On** |

**Add-ons → Secrets** → add a secret named `HF_TOKEN` with a **Write** token from
huggingface.co/settings/tokens. This keeps the token out of the notebook.

Then **Save Version → Save & Run All (Commit) → Save**. ~5–6 hours total.

## 1. Config — edit these two lines

In [ ]:
import os

REPO = "https://github.com/Ricky11234/fairness-aware-gpt2"
HF_REPO = "RealRick111/fairness-gpt2-qqp"   # your Hugging Face model repo

EPOCHS = 5
TRAIN_SIZE = 283011
LAMBDA_FAIR = 0.5
MODES = ["baseline", "cda", "cda_reg"]   # trained in this order

os.environ.update(EPOCHS=str(EPOCHS), TRAIN_SIZE=str(TRAIN_SIZE),
                  LAMBDA_FAIR=str(LAMBDA_FAIR))
print("will train:", MODES)

## 2. Fail fast — GPU, token, repo

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 x2 and re-commit."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {name} | compute {cap[0]}.{cap[1]}")
assert cap >= (7, 0), (
    f"{name} is compute {cap[0]}.{cap[1]}; PyTorch needs >= 7.0. "
    "P100 (6.0) has no kernels — switch to GPU T4 x2."
)
print("GPU OK")

In [ ]:
# Fetch the HF token from Kaggle Secrets now, so an unattended run doesn't get
# 5 hours in and then fail at the upload step on a missing credential.
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    assert HF_TOKEN and HF_TOKEN.startswith("hf_")
    print("HF_TOKEN found")
except Exception as e:
    raise SystemExit(
        "No usable HF_TOKEN. Add-ons > Secrets > add HF_TOKEN (a Write token from "
        f"huggingface.co/settings/tokens), then re-commit.  [{e}]"
    ) from e

## 3. Clone

In [ ]:
%cd /kaggle/working
!rm -rf fairness-repo
!git clone -q $REPO fairness-repo
%cd fairness-repo

In [ ]:
# identity.py generates the counterfactuals AND computes the flip rate, so stale
# code silently corrupts the headline number. Fail now, not in 5 hours.
from pathlib import Path

ident = Path("src/fairness_gpt2/identity.py").read_text()
assert "AMBIGUOUS_PRONOUNS" in ident and "subgroups_of" in ident, (
    "Cloned repo is missing recent fixes — you have unpushed changes. "
    "GitHub Desktop > Push origin, then re-commit."
)
print("Repo is current")

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19"

## 4. Data

In [ ]:
!PYTHONPATH=src python scripts/download_data.py --only qqp --train-size $TRAIN_SIZE --skip-test

In [ ]:
import sys

sys.path.insert(0, "src")
from fairness_gpt2.data import load_qqp

n_dev = len(load_qqp("data/quora-dev.csv"))
print("dev pairs:", n_dev)
assert n_dev == 40430, f"dev is {n_dev}, expected 40430"
print("Data OK")

## 5. Smoke test

One epoch on 2,000 pairs, ~2 min. Confirms the pipeline before the long runs.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

### Check
- **`device=cuda`** — if `cpu`, stop.
- **`fairness_loss` > 0** — if `0.0`, no counterfactuals were made.

## 6. Train all three

~1.5–2 h each. baseline → cda → cda_reg. Each writes
`results/reproduced/<mode>.json` as it finishes, so partial progress survives a
session cut.

In [ ]:
import subprocess

for mode in MODES:
    print(f"\n{'=' * 60}\nTRAINING {mode}\n{'=' * 60}", flush=True)
    cmd = [
        "python", "-m", "fairness_gpt2.train",
        "--mode", mode,
        "--train", "data/quora-train.csv",
        "--dev", "data/quora-dev.csv",
        "--out", f"/kaggle/working/checkpoints/{mode}",
        "--epochs", str(EPOCHS),
        "--save-half",
    ]
    if mode == "cda_reg":
        cmd += ["--lambda-fair", str(LAMBDA_FAIR)]
    env = {**os.environ, "PYTHONPATH": "src"}
    r = subprocess.run(cmd, env=env)
    if r.returncode != 0:
        print(f"WARNING: {mode} exited {r.returncode}; continuing with the rest.")

## 7. Upload cda_reg to Hugging Face

In [ ]:
from huggingface_hub import HfApi, create_repo, login

login(HF_TOKEN)
create_repo(HF_REPO, exist_ok=True)
HfApi().upload_folder(
    folder_path="/kaggle/working/checkpoints/cda_reg",
    repo_id=HF_REPO,
    commit_message="Fairness-aware GPT-2 paraphrase checkpoint (5 epochs)",
)
print(f"Uploaded raw files -> https://huggingface.co/{HF_REPO}")
print(f'Streamlit secret:  MODEL_REPO = "{HF_REPO}"')

**If your HF repo still has the old `pytorch_model.zip`, delete it** on the Hub
(click it → trash → commit). The app wants `pytorch_model.bin`, which this cell
just uploaded correctly.

## 8. The comparison — baseline vs fairness

In [ ]:
import json
from pathlib import Path

from fairness_gpt2.results import intervention_effect, load_reproduced

reproduced = load_reproduced()
print("trained models:", list(reproduced), "\n")

hdr = f"{'model':<12}{'dev_acc':>10}{'subgroup_gap':>14}{'flip_rate':>11}"
print(hdr); print("-" * len(hdr))
for mode in MODES:
    m = reproduced.get(mode)
    if m:
        print(f"{mode:<12}{m['accuracy']:>10.4f}{m['subgroup_gap']:>14.4f}{m['flip_rate']:>11.4f}")
    else:
        print(f"{mode:<12}{'(missing)':>10}")

eff = intervention_effect(reproduced)
if eff:
    f, a, g = eff["flip_rate"], eff["accuracy"], eff["subgroup_gap"]
    print(f"\n{'=' * 60}")
    print("BASELINE  ->  CDA + REGULARIZATION")
    print(f"{'=' * 60}")
    print(f"  flip rate     {f['baseline']:.4f} -> {f['cda_reg']:.4f}   ({f['pct']:+.0%})")
    print(f"  dev accuracy  {a['baseline']:.4f} -> {a['cda_reg']:.4f}   ({a['delta']:+.4f})")
    print(f"  subgroup gap  {g['baseline']:.4f} -> {g['cda_reg']:.4f}   ({g['pct']:+.0%})")
    print(f"\n  Identity robustness improved {abs(f['pct']):.0%} at "
          f"{'no cost to' if abs(a['delta']) < 0.005 else 'a small change in'} accuracy.")
    print(f"  Baseline flipped {eff['flips']['baseline']} of {eff['flips']['n_identity']:,} "
          f"identity pairs; CDA+Reg flips {eff['flips']['cda_reg']}.")
    if g["delta"] > 0:
        print(f"\n  Note: the regularizer widened the subgroup gap ({g['pct']:+.0%}) while "
              "cutting the flip rate. Different fairness notions; they don't move together.")
else:
    print("\nComparison needs both baseline and cda_reg — check the table above.")

## 9. Package results to download

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/reproduced", "zip", "results", "reproduced")
print("Download reproduced.zip from the Output panel, then:")
print("  unzip into your project's results/reproduced/, commit, push.")
print("  Your live app fills in the comparison automatically.")

## After it finishes

1. **Set the Streamlit secret** (if not already): app → Settings → Secrets →
   `MODEL_REPO = "RealRick111/fairness-gpt2-qqp"` → the app reboots and predictions work.
2. **Download `reproduced.zip`** from the Output panel → unzip into
   `results\reproduced\` → commit → push. The comparison section fills in.

Both `baseline.json` and `cda_reg.json` (and `cda.json`) will be in the zip, so
the "What the interventions buy" section goes live with your own numbers.